# Google Web Search Copyright Removals — Analysis

**Data source:** Google Transparency Report — [Web Search Copyright Removals](https://transparencyreport.google.com/copyright/overview)  
**Dataset URL:** https://storage.googleapis.com/transparencyreport/google-websearch-copyright-removals.zip  
**License:** Public data published by Google under their transparency report programme.

This notebook answers:
1. Who files the most copyright removal requests?
2. Which organisations act as intermediaries (reporting agents)?
3. Which domains are most targeted?
4. How have removal requests trended over time?
5. What fraction of requested URLs actually get removed?

## 1. Setup

In [ ]:
import io
import sys
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
import seaborn as sns

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

DATA_DIR = Path('../data')
DATASET_URL = (
    'https://storage.googleapis.com/transparencyreport/'
    'google-websearch-copyright-removals.zip'
)
DATA_DIR.mkdir(exist_ok=True)
print('Setup complete. Python', sys.version)

## 2. Download + extract dataset

In [ ]:
if not any(DATA_DIR.glob('*.csv')):
    print('Downloading dataset (~80 MB)…')
    resp = requests.get(DATASET_URL, timeout=120)
    resp.raise_for_status()
    (DATA_DIR / 'google-websearch-copyright-removals.zip').write_bytes(resp.content)
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        zf.extractall(DATA_DIR)
        print('Extracted:', zf.namelist())
else:
    print('Dataset already present, skipping download.')

csv_files = list(DATA_DIR.glob('*.csv'))
print('CSV files:', [p.name for p in csv_files])

## 3. Data loading + overview

In [ ]:
frames = {}
for p in csv_files:
    df = pd.read_csv(p, low_memory=False)
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    frames[p.stem] = df
    print(f'\n=== {p.name} ===')
    print('Shape:', df.shape)
    print('Columns:', df.columns.tolist())
    display(df.head(3))
    display(df.describe(include='all').T)

In [ ]:
# Work with the primary requests frame — pick the largest CSV
main_key = max(frames, key=lambda k: len(frames[k]))
df = frames[main_key].copy()
print(f'Using: {main_key} ({len(df):,} rows)')

# Date parsing
date_col = next((c for c in df.columns if 'date' in c), None)
if date_col:
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df = df.dropna(subset=[date_col])
    df['year'] = df[date_col].dt.year
    df['year_month'] = df[date_col].dt.to_period('M')
    print(f'Date range: {df[date_col].min().date()} → {df[date_col].max().date()}')
print(f'Records after date parse: {len(df):,}')

## 4. Top copyright owners

In [ ]:
owner_col = next(
    (c for c in df.columns if 'copyright_owner' in c or ('owner' in c and 'copyright' in c)),
    next((c for c in df.columns if 'owner' in c), None)
)
print('Owner column:', owner_col)

if owner_col:
    top_owners = df[owner_col].value_counts().head(20)
    fig, ax = plt.subplots(figsize=(10, 7))
    top_owners.sort_values().plot.barh(ax=ax)
    ax.set_title('Top 20 Copyright Owners by Removal Requests')
    ax.set_xlabel('Number of Requests')
    plt.tight_layout()
    plt.show()
    print('\nTop 5:')
    print(top_owners.head())

## 5. Top reporting organizations

In [ ]:
org_col = next(
    (c for c in df.columns if 'reporting' in c or 'requester' in c or 'reporter' in c),
    None
)
print('Reporting org column:', org_col)

if org_col:
    top_orgs = df[org_col].value_counts().head(20)
    fig, ax = plt.subplots(figsize=(10, 7))
    top_orgs.sort_values().plot.barh(ax=ax, color=sns.color_palette('muted')[1])
    ax.set_title('Top 20 Reporting Organizations')
    ax.set_xlabel('Number of Requests')
    plt.tight_layout()
    plt.show()
    print('\nTop 5:')
    print(top_orgs.head())

## 6. Most-targeted domains

In [ ]:
domain_col = next((c for c in df.columns if 'domain' in c), None)
print('Domain column:', domain_col)

if domain_col:
    top_domains = df[domain_col].value_counts().head(25)
    fig, ax = plt.subplots(figsize=(10, 8))
    top_domains.sort_values().plot.barh(ax=ax, color=sns.color_palette('muted')[2])
    ax.set_title('Top 25 Most-Targeted Domains')
    ax.set_xlabel('Removal Requests')
    plt.tight_layout()
    plt.show()
    print('\nTop 5 targeted domains:')
    print(top_domains.head())

## 7. Trends over time

In [ ]:
if date_col and 'year' in df.columns:
    yearly = df.groupby('year').size().rename('requests')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Yearly
    yearly.plot(ax=axes[0], marker='o', color=sns.color_palette('muted')[3])
    axes[0].set_title('Removal Requests per Year')
    axes[0].set_xlabel('Year')
    axes[0].set_ylabel('Requests')

    # Monthly (last 5 years)
    cutoff_year = df['year'].max() - 5
    recent = df[df['year'] > cutoff_year]
    monthly = recent.groupby('year_month').size().rename('requests')
    monthly.plot(ax=axes[1], color=sns.color_palette('muted')[4])
    axes[1].set_title(f'Monthly Requests (last 5 years)')
    axes[1].set_xlabel('Month')
    axes[1].set_ylabel('Requests')

    plt.tight_layout()
    plt.show()

    peak = yearly.idxmax()
    print(f'Peak year: {peak} with {int(yearly[peak]):,} requests')
    print(f'Year-on-year change (latest 2):\n{yearly.tail(2).pct_change().tail(1) * 100}')

## 8. URLs removed vs requested

In [ ]:
url_req_col = next(
    (c for c in df.columns if 'urls_requested' in c or 'requested_to_remove' in c
     or 'total_urls' in c or 'urls_asked' in c),
    None
)
url_rem_col = next(
    (c for c in df.columns if 'urls_removed' in c or 'actually_removed' in c
     or 'urls_deindexed' in c),
    None
)
print('Requested col:', url_req_col, '| Removed col:', url_rem_col)

if url_req_col and url_rem_col:
    df[url_req_col] = pd.to_numeric(df[url_req_col], errors='coerce')
    df[url_rem_col] = pd.to_numeric(df[url_rem_col], errors='coerce')
    valid = df[[url_req_col, url_rem_col]].dropna()

    total_req = int(valid[url_req_col].sum())
    total_rem = int(valid[url_rem_col].sum())
    rate = total_rem / total_req * 100 if total_req else 0

    print(f'Total URLs requested: {total_req:,}')
    print(f'Total URLs removed:   {total_rem:,}')
    print(f'Removal rate:         {rate:.1f}%')

    sample = valid.sample(min(8000, len(valid)), random_state=42)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(sample[url_req_col], sample[url_rem_col], alpha=0.3, s=8,
               color=sns.color_palette('muted')[5])
    ax.set_title('URLs Requested vs Removed (sample)')
    ax.set_xlabel('URLs Requested')
    ax.set_ylabel('URLs Removed')
    plt.tight_layout()
    plt.show()
else:
    print('URL count columns not found in this dataset; skipping removal-rate chart.')
    print('Available columns:', df.columns.tolist())

## 9. Key findings

Run all cells above first; the cell below summarises programmatically.

In [ ]:
findings = []
findings.append(f'Dataset: {len(df):,} removal requests')

if date_col:
    findings.append(
        f'Date range: {df[date_col].min().date()} to {df[date_col].max().date()}'
    )

if owner_col:
    top3_owners = df[owner_col].value_counts().head(3).index.tolist()
    findings.append(f'Top copyright owners: {top3_owners}')

if org_col:
    top3_orgs = df[org_col].value_counts().head(3).index.tolist()
    findings.append(f'Top reporting orgs: {top3_orgs}')

if domain_col:
    top3_domains = df[domain_col].value_counts().head(3).index.tolist()
    findings.append(f'Most targeted domains: {top3_domains}')

if date_col and 'year' in df.columns:
    yearly = df.groupby('year').size()
    peak = yearly.idxmax()
    findings.append(f'Peak year: {peak} ({int(yearly[peak]):,} requests)')

print('=== KEY FINDINGS ===')
for i, f in enumerate(findings, 1):
    print(f'  {i}. {f}')